# End-to-End Flood Detection Training

This notebook trains an end-to-end model: Clay encoder (frozen) + decoder + classifier.

Unlike the two-stage approach, this processes raw images directly without pre-computing embeddings.

In [ ]:
import os
from pathlib import Path

def find_project_root(marker='claymodel'):
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / marker).exists():
            return path
    raise FileNotFoundError(f"Project root not found")

os.chdir(find_project_root())
print(f"Working directory: {os.getcwd()}")

In [ ]:
from lightning.pytorch import Trainer, seed_everything
from lightning.pytorch.cli import LightningArgumentParser, instantiate_class
import lightning.pytorch as L
from claymodel.finetune.flood_detection.end_to_end_flood_classifier import EndToEndFloodClassifier
from claymodel.finetune.flood_detection.end_to_end_gfm_datamodule import EndToEndGFMDataModule
from datetime import datetime

seed_everything(42)

In [ ]:
def train_end_to_end_model(config_path, test_after_training=False):
    objects = ["callbacks", "logger", "plugins"]
    parser = LightningArgumentParser()
    parser.add_lightning_class_args(EndToEndFloodClassifier, "model")
    parser.add_lightning_class_args(EndToEndGFMDataModule, "data")
    parser.add_lightning_class_args(Trainer, "trainer")
    config = parser.parse_path(config_path)
    trainer_config = dict(config["trainer"])
    
    for object_type in objects:
        if object_type in trainer_config and trainer_config[object_type]:
            instantiated = []
            for item_config in trainer_config[object_type]:
                if hasattr(item_config, 'class_path') and hasattr(item_config, 'init_args'):
                    if object_type == "logger":
                        if item_config.class_path == "lightning.pytorch.loggers.CSVLogger":
                            item_config.init_args['version'] = datetime.now().strftime("%Y%m%d_%H%M%S")
                        elif item_config.class_path == "lightning.pytorch.loggers.WandbLogger":
                            item_config.init_args['name'] = "EndToEnd_" + datetime.now().strftime("%Y%m%d_%H%M%S")
                    if object_type == "callbacks" and item_config.class_path == "lightning.pytorch.callbacks.ModelCheckpoint":
                        item_config.init_args['dirpath'] = os.path.join(item_config.init_args['dirpath'], datetime.now().strftime("%Y%m%d_%H%M%S"))
                    item = instantiate_class((), item_config)
                    instantiated.append(item)
                elif isinstance(item_config, dict) and "class_path" in item_config:
                    item = instantiate_class((), item_config)
                    instantiated.append(item)
                else:
                    instantiated.append(item_config)
            trainer_config[object_type] = instantiated
    
    model = EndToEndFloodClassifier(**config["model"])
    datamodule = EndToEndGFMDataModule(**config["data"])
    trainer = Trainer(**trainer_config)
    
    try:
        model_config = dict(config["model"])
        for logger in trainer.loggers:
            if isinstance(logger, L.pytorch.loggers.WandbLogger):
                run = logger.experiment
                if run is not None:
                    run.config.update({"model": model_config}, allow_val_change=True)
            elif isinstance(logger, L.pytorch.loggers.CSVLogger):
                logger.log_hyperparams({"model": model_config})
    except Exception as e:
        print(f"Warning: {e}")
    
    result = trainer.fit(model, datamodule)
    if test_after_training:
        result = trainer.test(model, datamodule)
    print("Results:", result)
    return model, datamodule, trainer, result

## Train the Model

In [ ]:
CONFIG_PATH = "configs/train_end_to_end_flood_detection.yaml"
model, datamodule, trainer, result = train_end_to_end_model(CONFIG_PATH, test_after_training=True)